In [ ]:
import open3d as o3d

# Path to your PLY file
ply_path = "/storage/jothi/Yash/Datasets/correct_instance_labeled datasets_with_normals/Bridge CG 01_instance_cloud_crop_0.ply"

# Load the point cloud
pcd = o3d.io.read_point_cloud(ply_path)

# Print basic information
print(pcd)
print("Number of points:", len(pcd.points))

# Visualize
o3d.visualization.draw_geometries([pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PointCloud with 40000 points.
Number of points: 40000
[Open3D WARNING] GLFW Error: Failed to detect any supported platform
[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] GLFW Error: OSMesa: Library not found
[Open3D WARNING] Failed to create window
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.


In [3]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

pcd = o3d.io.read_point_cloud("/storage/jothi/Yash/Datasets/correct_instance_labeled datasets_with_normals/Bridge CG 01_instance_cloud_crop_0.ply")

points = np.asarray(pcd.points)

# Use RGB colours if available
if pcd.has_colors():
    colors = np.asarray(pcd.colors)
    rgb = (colors * 255).astype(np.uint8)
    marker_color = [
        f"rgb({r},{g},{b})" for r, g, b in rgb
    ]
else:
    marker_color = "royalblue"

fig = go.Figure(data=[
    go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode="markers",
        marker=dict(
            size=2,
            color=marker_color,
            opacity=0.8
        )
    )
])

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    title=f"Point Cloud: {len(points):,} points",
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [4]:
import open3d as o3d
import numpy as np
from plyfile import PlyData

path = "/storage/jothi/Yash/Datasets/correct_instance_labeled datasets_with_normals/Bridge CG 01_instance_cloud_crop_0.ply"

# Inspect PLY properties
ply = PlyData.read(path)
vertex = ply["vertex"]

print("Number of points:", len(vertex.data))
print("Available properties:")
print(vertex.data.dtype.names)

# Load geometry
pcd = o3d.io.read_point_cloud(path)
print("\nOpen3D point cloud:", pcd)
print("Has RGB colours:", pcd.has_colors())
print("Has normals:", pcd.has_normals())

Number of points: 40000
Available properties:
('x', 'y', 'z', 'red', 'green', 'blue', 'class_id', 'instance_id', 'nx', 'ny', 'nz', 'z_rel')

Open3D point cloud: PointCloud with 40000 points.
Has RGB colours: True
Has normals: True


In [5]:
import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative
from plyfile import PlyData

path = "/storage/jothi/Yash/Datasets/correct_instance_labeled datasets_with_normals/Bridge CG 01_instance_cloud_crop_0.ply"

# Load PLY data
data = PlyData.read(path)["vertex"].data

xyz = np.column_stack([data["x"], data["y"], data["z"]])
rgb = np.column_stack([data["red"], data["green"], data["blue"]])
class_ids = np.asarray(data["class_id"])
instance_ids = np.asarray(data["instance_id"])

# Convert RGB to 0-255 if necessary
if rgb.max() <= 1:
    rgb = rgb * 255
rgb = np.clip(rgb, 0, 255).astype(np.uint8)

# Generate a distinct colour for each unique label
palette = qualitative.Alphabet + qualitative.Dark24 + qualitative.Light24

def make_colors(labels):
    unique_labels = np.unique(labels)
    color_map = {
        label: palette[i % len(palette)]
        for i, label in enumerate(unique_labels)
    }
    return [color_map[label] for label in labels], color_map

semantic_colors, semantic_map = make_colors(class_ids)
instance_colors, instance_map = make_colors(instance_ids)

original_colors = [
    f"rgb({r},{g},{b})" for r, g, b in rgb
]

# Create interactive figure
fig = go.Figure()

for name, colors, visible in [
    ("Original RGB", original_colors, True),
    ("Semantic classes", semantic_colors, False),
    ("Instance IDs", instance_colors, False),
]:
    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0],
        y=xyz[:, 1],
        z=xyz[:, 2],
        mode="markers",
        marker=dict(size=2, color=colors, opacity=1),
        name=name,
        visible=visible,
        hovertemplate=(
            "X: %{x:.3f}<br>"
            "Y: %{y:.3f}<br>"
            "Z: %{z:.3f}<extra></extra>"
        )
    ))

fig.update_layout(
    title="Bridge CG 01: Point Cloud Labels",
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    updatemenus=[dict(
        buttons=[
            dict(
                label=name,
                method="update",
                args=[
                    {"visible": [i == j for i in range(3)]},
                    {"title": name}
                ]
            )
            for j, name in enumerate([
                "Original RGB",
                "Semantic classes",
                "Instance IDs"
            ])
        ],
        direction="down",
        x=0.02,
        y=0.98
    )],
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()